In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Get Data

In [7]:
# import the databases
food = pd.read_csv("data/FoodData_Central_foundation_food_csv_2026-10-29/food.csv")
food_survey = pd.read_csv("data/FoodData_Central_survey_food_csv_2024-10-31/food.csv")
nutrient_defs = pd.read_csv("data/FoodData_Central_Supporting_Data_csv_2021-04-28/nutrient.csv")
food_nutrient_survey = pd.read_csv("data/FoodData_Central_survey_food_csv_2024-10-31/food_nutrient.csv")

# Fixed variable name: nutrient_defs instead of nutrients
print(len(food), len(food_survey), len(nutrient_defs), len(food_nutrient_survey))

# Merge foundation and survey food files using pd.concat
food = pd.concat([food, food_survey], ignore_index=True).dropna(subset=["description"])
food["description"] = food["description"].str.lower()
print(f"Combined food rows: {len(food)}")

# Map nutrient names onto the survey food nutrients table using the lookup definitions
food_nutrient_survey["nutrient_name"] = food_nutrient_survey["nutrient_id"].map(
    nutrient_defs.set_index("id")["name"]
).str.lower()
print(f"Processed food nutrient rows: {len(food_nutrient_survey)}")

101532 5432 474 353015
Combined food rows: 106956
Processed food nutrient rows: 353015


In [9]:
food_nutrient_survey.columns

Index(['id', 'fdc_id', 'nutrient_id', 'amount', 'data_points', 'derivation_id',
       'min', 'max', 'median', 'footnote', 'min_year_acquired',
       'nutrient_name'],
      dtype='str')

In [10]:
len(food_nutrient_survey)

353015

In [11]:
food.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"hummus, sabra classic",16.0,2019-04-01
1,319875,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
2,319876,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
3,319877,sub_sample_food,hummus,16.0,2019-04-01
4,319878,sub_sample_food,hummus,16.0,2019-04-01


In [12]:
food_nutrient_survey.head

<bound method NDFrame.head of               id   fdc_id  nutrient_id   amount  data_points  derivation_id  \
0       34136169  2705384          301  125.000          NaN            NaN   
1       34136178  2705384          319   57.000          NaN            NaN   
2       34136185  2705384          337    0.000          NaN            NaN   
3       34136218  2705384          628    0.002          NaN            NaN   
4       34136219  2705384          629    0.000          NaN            NaN   
...          ...      ...          ...      ...          ...            ...   
353010  34489112  2710814          208  892.000          NaN            NaN   
353011  34489151  2710814          601    0.000          NaN            NaN   
353012  34489135  2710814          337    0.000          NaN            NaN   
353013  34489121  2710814          304    0.000          NaN            NaN   
353014  34489143  2710814          418    0.000          NaN            NaN   

        min  max  med

In [13]:
# How many unique
unique_descriptions = food["description"].unique()
len(unique_descriptions)

17159

In [14]:
unique_descriptions[:10]

<StringArray>
['hummus, sabra classic',                'hummus',         'hummus, other',
    'hummus - nfy12140o',    'hummus - nfy12140p',    'hummus - nfy12140q',
    'hummus - nfy12140r',    'hummus - nfy12140s',    'hummus - nfy12140f',
    'hummus - nfy12140g']
Length: 10, dtype: str

In [16]:
unique_descriptions[-10:]

<StringArray>
[                'celery, cooked, as ingredient',
 'dark green vegetables as ingredient in omelet',
              'tomatoes as ingredient in omelet',
      'other vegetables as ingredient in omelet',
               'mirepoix, cooked, as ingredient',
             'vegetables as ingredient in curry',
             'vegetables as ingredient in soups',
             'vegetables as ingredient in stews',
             'sauce as ingredient in hamburgers',
          'industrial oil as ingredient in food']
Length: 10, dtype: str

In [17]:
# Find random indexes of food to explore
import random
random_number = random.randint(0, len(unique_descriptions)-10)
unique_descriptions[random_number:random_number+10]

<StringArray>
[                                   'meaures & comp, raw, rep 5, chicken, thigh, enhanced (ok1)',
                                    'meaures & comp, raw, rep 4, chicken, thigh, enhanced (mi1)',
                                    'meaures & comp, raw, rep 1, chicken, thigh, enhanced (ok1)',
         'thiamin, chicken, dark meat, lean only, thigh, raw, enhanced (ok1,mi1) - 14b-07-06-b1',
    'vitamin b12, chicken, dark meat, lean only, thigh, raw, enhanced (ok1,mi1) - 14b-07-06-b12',
      'riboflavin, chicken, dark meat, lean only, thigh, raw, enhanced (ok1,mi1) - 14b-07-06-b2',
          'niacin, chicken, dark meat, lean only, thigh, raw, enhanced (ok1,mi1) - 14b-07-06-b3',
      'vitamin b6, chicken, dark meat, lean only, thigh, raw, enhanced (ok1,mi1) - 14b-07-06-b6',
 'cholesterol,  chicken, dark meat, lean only, thigh, raw, enhanced (ok1,mi1) - 14b-07-06-chttu',
      'fa, chicken, dark meat, lean only, thigh, raw, enhanced (ok1,mi1) - 14b-07-06-fattyacids']
Length

### Food Categories

In [18]:
food.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"hummus, sabra classic",16.0,2019-04-01
1,319875,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
2,319876,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
3,319877,sub_sample_food,hummus,16.0,2019-04-01
4,319878,sub_sample_food,hummus,16.0,2019-04-01


In [19]:
unique_categories = food["food_category_id"].unique()
len(unique_categories)

192

In [20]:
food["food_category_id"].value_counts()

food_category_id
11.0      17641
9.0       12448
1.0       10970
20.0       9594
16.0       8744
          ...  
7804.0        2
9602.0        1
6006.0        1
7704.0        1
9204.0        1
Name: count, Length: 191, dtype: int64

In [21]:
# Get the food categories
food_cats = pd.read_csv("data/FoodData_Central_Supporting_Data_csv_2021-04-28/food_category.csv")
food_cats

,id,code,description
0,1,100,Dairy and Egg Products
1,2,200,Spices and Herbs
2,3,300,Baby Foods
3,4,400,Fats and Oils
4,5,500,Poultry Products
5,6,600,"Soups, Sauces, and Gravies"
6,7,700,Sausages and Luncheon Meats
7,8,800,Breakfast Cereals
8,9,900,Fruits and Fruit Juices
9,10,1000,Pork Products


### 10 Foods we want

To keep the data simple, we will reduce the databases from FoodData Central to 10 different foods.

> Why these foods?

Because we have images for these foods ready to go.

In [22]:
ten_whole_foods = ['apple',
                  'banana',
                  'beef', # steak etc
                  'blueberries',
                  'carrots',
                  'chicken_wings',
                  'egg', # Whole eggs
                  'honey',
                  'mushrooms',
                  'strawberries']

ten_whole_foods

['apple',
 'banana',
 'beef',
 'blueberries',
 'carrots',
 'chicken_wings',
 'egg',
 'honey',
 'mushrooms',
 'strawberries']

In [23]:
food.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"hummus, sabra classic",16.0,2019-04-01
1,319875,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
2,319876,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
3,319877,sub_sample_food,hummus,16.0,2019-04-01
4,319878,sub_sample_food,hummus,16.0,2019-04-01


In [24]:
# Foundation food is the ground truth for a certain type of food, excludes some detail about the food
# E.g the data_type foundation_food for chicken with the original unique ID for chicken
foundation_food = food[(food["data_type"] == "foundation_food") | (food["data_type"] == "survey_fndds_food")]
len(foundation_food)

5930

In [25]:
foundation_food[foundation_food["description"].str.contains("blue")]

,fdc_id,data_type,description,food_category_id,publication_date
42186,2263889,foundation_food,"blueberries, raw",9.0,2022-04-28
43441,2346411,foundation_food,"blueberries, raw",9.0,2022-10-28
60660,2684446,foundation_food,"crustaceans, crab, blue swimming, lump, pasteu...",15.0,2024-04-18
101854,2705705,survey_fndds_food,"cheese, blue or roquefort",1602.0,2022-10-28
104147,2707998,survey_fndds_food,"pie, blueberry",5502.0,2022-10-28
105347,2709198,survey_fndds_food,"blueberries, dried",6016.0,2022-10-28
105424,2709275,survey_fndds_food,"blueberries, raw",6011.0,2022-10-28
105426,2709277,survey_fndds_food,"blueberries, frozen",6011.0,2022-10-28
105427,2709278,survey_fndds_food,blueberry pie filling,8806.0,2022-10-28
105472,2709323,survey_fndds_food,blueberry juice,7006.0,2022-10-28


In [26]:
foundation_foods = foundation_food["description"]
foundation_foods[20:40]

4153               peanut butter, smooth style, with salt
4329                             cheese, parmesan, grated
4491    cheese, pasteurized process, american, vitamin...
4580    grapefruit juice, white, canned or bottled, un...
4723                                 peaches, yellow, raw
4817    seeds, sunflower seed kernels, dry roasted, wi...
4951      sausage, italian, pork, mild, cooked, pan-fried
5164                  bread, white, commercially prepared
5285          sausage, turkey, breakfast links, mild, raw
5428                                        cheese, swiss
5489    kale, frozen, cooked, boiled, drained, without...
5751    carrots, frozen, unprepared (includes foods fo...
5991                            mustard, prepared, yellow
6198                                figs, dried, uncooked
6339                                kiwifruit, green, raw
6491                              melons, cantaloupe, raw
6650                                      nectarines, raw
6794    orange

In [28]:
# Found a list of foundation foods we're going to start with!
foundation_foods_list = list(foundation_foods)
for food in foundation_foods_list:
    if "blue" in food:
        print(food)

blueberries, raw
blueberries, raw
crustaceans, crab, blue swimming, lump, pasteurized, refrigerated
cheese, blue or roquefort
pie, blueberry
blueberries, dried
blueberries, raw
blueberries, frozen
blueberry pie filling
blueberry juice
blue or roquefort cheese dressing
blue or roquefort cheese dressing, light
blue or roquefort cheese dressing, fat free
blueberry syrup


In [33]:
#food.loc[(food["description"].str.contains("chicken", case=False)) & (food["description"].str.contains("drumstick", case=False))][-10]
# find chicken in foundation food
for food in foundation_foods:
    if "chicken" in food.lower():
        print(food)

chicken, broilers or fryers, drumstick, meat only, cooked, braised
chicken, broiler or fryers, breast, skinless, boneless, meat only, cooked, braised
chicken, ground, with additives, raw
chicken, breast, boneless, skinless, raw
chicken, thigh, boneless, skinless, raw
chicken, drumstick, meat and skin, raw
chicken, thigh, meat and skin, raw
chicken, wing, meat and skin, raw
chicken, breast, meat and skin, raw
lunchmeat, chicken breast, sliced
mock chicken legs
chicken, ns as to part and cooking method, ns as to skin eaten
chicken, ns as to part and cooking method, skin eaten
chicken, ns as to part and cooking method, skin not eaten
chicken, ns as to part, baked, broiled, or roasted, ns as to skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin not eaten
chicken, ns as to part, rotisserie, ns as to skin eaten
chicken, ns as to part, rotisserie, skin eaten
chicken, ns as to part, rotisserie, skin not eaten
chick

In [37]:
# Reload foundation food dataset
food = pd.read_csv("data/FoodData_Central_foundation_food_csv_2026-10-29/food.csv")

# Run search again
chicken_wing_id = int(food.loc[food["description"].str.contains("Chicken", case=False)].iloc[0]["fdc_id"])
chicken_wing_id

330870

In [42]:
# 1. Search specifically for chicken wing (and fix the dtype warning)
food_nutrient_foundation = pd.read_csv(
    "data/FoodData_Central_foundation_food_csv_2026-10-29/food_nutrient.csv", 
    low_memory=False
)

# 2. Find a more specific item (e.g., 'chicken, wing')
chicken_row = food.loc[food["description"].str.contains("chicken, wing", case=False)].iloc[0]
chicken_wing_id = int(chicken_row["fdc_id"])

print(f"Selected Food: {chicken_row['description']} (FDC ID: {chicken_wing_id})")

# 3. Query the nutrient table for this specific ID
food_nutrient_foundation[food_nutrient_foundation["fdc_id"] == chicken_wing_id]

Selected Food: Chicken, wing, meat and skin, raw (FDC ID: 2727568)


,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired
151745,34678474,2727568,1002,2.945000,8.0,1.0,2.79100,3.1040,2.9360,NaN,2024.0
151746,34678468,2727568,1091,132.800000,8.0,1.0,126.00000,145.0000,131.5000,NaN,2024.0
151747,34678477,2727568,1007,0.752500,8.0,1.0,0.69000,0.8500,0.7350,NaN,2024.0
151748,34678473,2727568,1098,0.010530,8.0,1.0,0.00000,0.0324,0.0000,NaN,2024.0
151749,34678479,2727568,1087,13.550000,8.0,1.0,9.38000,23.3000,11.4000,NaN,2024.0
151750,34678472,2727568,1095,1.179000,8.0,1.0,1.09000,1.3100,1.1650,NaN,2024.0
151751,34678467,2727568,1090,16.530000,8.0,1.0,15.30000,17.7000,16.4500,NaN,2024.0
151752,34678475,2727568,1004,10.640000,8.0,1.0,8.97000,13.1500,10.5800,NaN,2024.0
151753,34678469,2727568,1092,194.000000,8.0,1.0,174.00000,233.0000,185.0000,NaN,2024.0
151754,34678478,2727568,1051,70.660000,8.0,1.0,69.38000,71.5100,70.9100,NaN,2024.0


### Get protein, carb, fat IDs

See this document for info on foundation foods and their nutrients

https://fdc.nal.usda.gov/docs/Foundation_Foods_Documetation_Apr2021.pdf

* Carbohydrate, by diffrence = total carbohydrates

In [48]:
nutrient_defs[(nutrient_defs["name"].str.contains("protein", case=False)) | \
         (nutrient_defs["name"].str.contains("carbohydrate", case=False)) | \
         (nutrient_defs["name"].str.contains("fat", case=False))]

,id,name,unit_name,nutrient_nbr,rank
4,1003,Protein,G,203.0,600.0
5,1004,Total lipid (fat),G,204.0,800.0
6,1005,"Carbohydrate, by difference",G,205.0,1110.0
50,1049,"Solids, non-fat",G,253.0,999999.0
51,1050,"Carbohydrate, by summation",G,205.2,1120.0
54,1053,Adjusted Protein,G,257.0,700.0
73,1072,"Carbohydrate, other",G,284.0,NaN
86,1085,Total fat (NLEA),G,298.0,900.0
258,1257,"Fatty acids, total trans",G,605.0,15400.0
259,1258,"Fatty acids, total saturated",G,606.0,9700.0


In [49]:
target_nutrients = nutrient_defs[nutrient_defs["name"].isin(["Protein", "Total lipid (fat)", "Carbohydrate, by difference"])]
target_nutrients

,id,name,unit_name,nutrient_nbr,rank
4,1003,Protein,G,203.0,600.0
5,1004,Total lipid (fat),G,204.0,800.0
6,1005,"Carbohydrate, by difference",G,205.0,1110.0


In [50]:
target_nutrients_dict = {1003: "protein",
                        1004: "fat",
                        1005: "carbohydrate"
}

### Get target food protein, fat, carbohydrates

We want to know the index on the target foods and the target nutrients and retrieve their values for each food/nutrient

E.g. 

{"food_1": {"protein": 100,
            "carbohydrate": 50,
            "fat": 20},
 "food_2": .......

 ...}

In [51]:
list(target_nutrients_dict.keys())

[1003, 1004, 1005]

In [52]:
food_nutrient

,id,name,unit_name,nutrient_nbr,rank
0,2047,Energy (Atwater General Factors),KCAL,957.0,280.0
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0
2,1001,Solids,G,201.0,200.0
3,1002,Nitrogen,G,202.0,500.0
4,1003,Protein,G,203.0,600.0
...,...,...,...,...,...
469,2061,"Ergosta-7,22-dienol",MG,NaN,16211.0
470,2062,"Ergosta-5,7-dienol",MG,NaN,16211.0
471,2063,Verbascose,G,NaN,2450.0
472,2064,Oligosaccharides,MG,NaN,2250.0


In [61]:
# Check the actual IDs for protein, fat, and carbs in your nutrient definitions table
#nutrient_defs[nutrient_defs["name"].str.contains("protein|total lipid|carbohydrate", case=False, na=False)]

In [62]:
nutrient_defs.dtypes

id                int64
name                str
unit_name           str
nutrient_nbr    float64
rank            float64
dtype: object

In [63]:
sorted(list(foundation_foods))

['abalone',
 'adobo, with noodles',
 'adobo, with rice',
 'agave liquid sweetener',
 'alaska pollock, raw',
 'alcoholic coffee drink',
 'alcoholic malt beverage',
 'alcoholic malt beverage, sweetened',
 'alfalfa sprouts, raw',
 'alfredo sauce',
 'alfredo sauce with added vegetables',
 'alfredo sauce with meat',
 'alfredo sauce with meat and added vegetables',
 'alfredo sauce with poultry',
 'alfredo sauce with poultry and added vegetables',
 'alfredo sauce with seafood',
 'alfredo sauce with seafood and added vegetables',
 'almond butter',
 'almond butter and jelly sandwich, on wheat bread',
 'almond butter and jelly sandwich, on white bread',
 'almond butter sandwich, on wheat bread',
 'almond butter sandwich, on white bread',
 'almond butter, creamy',
 'almond butter, lower sodium',
 'almond chicken',
 'almond milk, chocolate',
 'almond milk, nfs',
 'almond milk, sweetened',
 'almond milk, unsweetened',
 'almond milk, unsweetened, plain, refrigerated',
 'almond milk, unsweetened, pla

In [65]:
ten_whole_foods = ["chicken_wings",
    "apple",
    "banana",
    "beef", # steak, etc
    "carrots",
    "egg", # whole egg
    "strawberries",
    "blueberries",
    "mushrooms",
    "honey"
]

In [66]:
ten_whole_foods

['chicken_wings',
 'apple',
 'banana',
 'beef',
 'carrots',
 'egg',
 'strawberries',
 'blueberries',
 'mushrooms',
 'honey']

### Get ten whole foods `food_id`

Everything except blueberris and honey are available in `foundation_food`

For blueberry and honey, we'll have to dig into the survey data:

`data_exploration/data/FoodData_Central_Survey_food_csv_2020-10-30`

In [67]:
# Get all food ids from the foundation_food (honey ad blueberries in another dataset)
target_whole_foods = ['apple' # removed chicken wings... can come back later,
                     'banana',
                     'beef',
                     'blueberries',
                     'carrots',
                     'chicken',
                     'egg',
                     'honey',
                     'strawberries',
                     'mushrooms']

In [68]:
# str.contains can search on regex - https://stackoverflow.com/a/17973255/7900723
pattern = "|".join([f"(?i){food}" for food in target_whole_foods])
pattern

'(?i)applebanana|(?i)beef|(?i)blueberries|(?i)carrots|(?i)chicken|(?i)egg|(?i)honey|(?i)strawberries|(?i)mushrooms'

In [71]:
# Build a clean regex pattern with the case-insensitive flag at the front
pattern = "(?i)" + "|".join(target_whole_foods)

# Run with regex=True (which is the default)
foundation_food[
    foundation_food["description"].str.contains(pattern, regex=True)
].sort_values(by=["description"])

,fdc_id,data_type,description,food_category_id,publication_date
102946,2706797,survey_fndds_food,almond chicken,3404.0,2022-10-28
103640,2707491,survey_fndds_food,"almonds, honey roasted",2804.0,2022-10-28
20547,1750343,foundation_food,"apples, honeycrisp, with skin, raw",9.0,2020-10-30
20191,1105547,foundation_food,"apples, honeycrisp, with skin, raw",9.0,2020-10-30
102966,2706817,survey_fndds_food,asian chicken or turkey garden salad with cris...,3004.0,2022-10-28
...,...,...,...,...,...
102813,2706664,survey_fndds_food,"venison or deer, noodles, and vegetables inclu...",3002.0,2022-10-28
102812,2706663,survey_fndds_food,"venison or deer, potatoes, and vegetables excl...",3002.0,2022-10-28
102811,2706662,survey_fndds_food,"venison or deer, potatoes, and vegetables incl...",3002.0,2022-10-28
103680,2707531,survey_fndds_food,"walnuts, excluding honey roasted",2804.0,2022-10-28


In [72]:
foundation_food[foundation_food["description"].str.contains("honey")]

,fdc_id,data_type,description,food_category_id,publication_date
20191,1105547,foundation_food,"apples, honeycrisp, with skin, raw",9.0,2020-10-30
20547,1750343,foundation_food,"apples, honeycrisp, with skin, raw",9.0,2020-10-30
63976,2710816,foundation_food,"melons, honeydew, raw",9.0,2024-10-31
103640,2707491,survey_fndds_food,"almonds, honey roasted",2804.0,2022-10-28
103647,2707498,survey_fndds_food,"cashews, honey roasted",2804.0,2022-10-28
103660,2707511,survey_fndds_food,"mixed nuts, honey roasted",2804.0,2022-10-28
103669,2707520,survey_fndds_food,"peanuts, honey roasted",2804.0,2022-10-28
103674,2707525,survey_fndds_food,"pecans, honey roasted",2804.0,2022-10-28
103680,2707531,survey_fndds_food,"walnuts, excluding honey roasted",2804.0,2022-10-28
103681,2707532,survey_fndds_food,"walnuts, honey roasted",2804.0,2022-10-28


In [73]:
chicken_wing_id

2727568

In [74]:
# Map foods to food_id (these have been filtered from larger quantities to smaller quantities)
# For example, if there were 5 kinds of apple, only one was chosen
whole_foods_id_map = {1750339: "apple", # red delicious
    1105314: "banana", # Bananas, ripe and slightly ripe, raw
    1102702: "blueberries", # blueberries, raw	
    746763: "beef", # t-bone steak 
    746764: "carrots", # frozen unprepared
    331897: "chicken_wings", # Chicken, broilers or fryers, drumstick, meat o...	
    329490: "egg", # Egg, whole, dried	
    1103956: "honey", # Honey
    1750347: "mushrooms", # Mushrooms, white button
    747448: "strawberries" # strawberries, raw
}

In [75]:
list(whole_foods_id_map)

[1750339,
 1105314,
 1102702,
 746763,
 746764,
 331897,
 329490,
 1103956,
 1750347,
 747448]

In [81]:
# Find nutrition for eight whole foods using the foundation nutrient dataset
target_whole_foods_df = food_nutrient_foundation[
    (food_nutrient_foundation["fdc_id"].isin(list(whole_foods_id_map.keys()))) & 
    (food_nutrient_foundation["nutrient_id"].astype(str).isin([str(k) for k in target_nutrients_dict.keys()]))
][["fdc_id", "nutrient_id", "amount"]]
target_whole_foods_df

,fdc_id,nutrient_id,amount
34317,329490,1004,39.800000
34318,329490,1005,1.870000
34322,329490,1003,48.100000
41750,331897,1004,5.950000
41782,331897,1003,23.900000
41793,331897,1005,0.000000
71149,746763,1003,27.300000
71176,746763,1005,0.000000
71194,746763,1004,11.400000
71274,746764,1004,0.470000


In [82]:
# Pivot the table to how we want
target_whole_foods_df = target_whole_foods_df.pivot_table("amount", "fdc_id", "nutrient_id")
target_whole_foods_df

nutrient_id,1003,1004,1005
fdc_id,,,
329490,48.100000,39.8000,1.870000
331897,23.900000,5.9500,0.000000
746763,27.300000,11.4000,0.000000
746764,0.810000,0.4700,7.920000
747448,0.640000,0.2200,7.630000
1105314,0.740000,0.2900,23.000000
1750339,0.187500,0.2125,14.781700
1750347,2.890625,0.3708,4.079375


In [83]:
len(whole_foods_id_map)

10

In [84]:
target_whole_foods_df = target_whole_foods_df.reset_index(drop=False).rename_axis(None, axis=1)
target_whole_foods_df

,fdc_id,1003,1004,1005
0,329490,48.100000,39.8000,1.870000
1,331897,23.900000,5.9500,0.000000
2,746763,27.300000,11.4000,0.000000
3,746764,0.810000,0.4700,7.920000
4,747448,0.640000,0.2200,7.630000
5,1105314,0.740000,0.2900,23.000000
6,1750339,0.187500,0.2125,14.781700
7,1750347,2.890625,0.3708,4.079375


In [85]:
target_nutrients_dict

{1003: 'protein', 1004: 'fat', 1005: 'carbohydrate'}

In [87]:
# Rename the columns
target_whole_foods_df.rename(columns=target_nutrients_dict, inplace=True)
target_whole_foods_df

,fdc_id,protein,fat,carbohydrate
0,329490,48.100000,39.8000,1.870000
1,331897,23.900000,5.9500,0.000000
2,746763,27.300000,11.4000,0.000000
3,746764,0.810000,0.4700,7.920000
4,747448,0.640000,0.2200,7.630000
5,1105314,0.740000,0.2900,23.000000
6,1750339,0.187500,0.2125,14.781700
7,1750347,2.890625,0.3708,4.079375


In [88]:
# Add food names
target_whole_foods_df["food_name"] = target_whole_foods_df["fdc_id"].map(whole_foods_id_map)
target_whole_foods_df

,fdc_id,protein,fat,carbohydrate,food_name
0,329490,48.100000,39.8000,1.870000,egg
1,331897,23.900000,5.9500,0.000000,chicken_wings
2,746763,27.300000,11.4000,0.000000,beef
3,746764,0.810000,0.4700,7.920000,carrots
4,747448,0.640000,0.2200,7.630000,strawberries
5,1105314,0.740000,0.2900,23.000000,banana
6,1750339,0.187500,0.2125,14.781700,apple
7,1750347,2.890625,0.3708,4.079375,mushrooms


### All amounts are per 100g

### Export first 10 target food nutrition information

In [89]:
target_whole_foods_df.to_csv("target_ten_whole_food_nutrition_info.csv", index=False)

In [90]:
ten_whole_foods

['chicken_wings',
 'apple',
 'banana',
 'beef',
 'carrots',
 'egg',
 'strawberries',
 'blueberries',
 'mushrooms',
 'honey']